In [0]:
%pip install langchain langchain-community langchain-core \
             langchain-text-splitters langchain-databricks \
             pymupdf sentence-transformers

INFO: pip is looking at multiple versions of langchain-databricks to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-databricks to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of langchain-text-splitters to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-text-splitters to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constr

In [0]:
dbutils.library.restartPython()

In [0]:
# ---- imports.py ----
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from langchain_core.documents import Document
from pyspark.sql import SparkSession
import pandas as pd
import json, os, time

print("✅ All imports successful")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-093197c3-7a9d-4b65-8835-70a69ec062a7/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


✅ All imports successful


In [0]:
# ---- secrets.py ----

# Load from Databricks secrets
SARVAM_API_KEY = dbutils.secrets.get(scope="api_keys_scope", key="sarvam_api_key")
OPENAI_API_KEY = dbutils.secrets.get(scope="api_keys_scope", key="openai_api_key")

# Set as environment variables so LangChain picks them up automatically
import os
os.environ["OPENAI_API_KEY"]  = OPENAI_API_KEY
os.environ["SARVAM_API_KEY"]  = SARVAM_API_KEY

print("✅ API keys loaded")

✅ API keys loaded


In [0]:
# ---- config.py ----
PDF_DIR  = "/Volumes/export_ai/rag/legal_docs"
JSON_DIR = "/Volumes/export_ai/rag/country_guidelines"

ENDPOINT_NAME = "export_ai_vector_endpoint"
INDEX_NAME    = "export_ai.rag.document_chunks_index"
TABLE_NAME    = "export_ai.rag.document_chunks"

print("✅ Config loaded")
print(f"PDF_DIR  : {PDF_DIR}")
print(f"JSON_DIR : {JSON_DIR}")

✅ Config loaded
PDF_DIR  : /Volumes/export_ai/rag/legal_docs
JSON_DIR : /Volumes/export_ai/rag/country_guidelines


In [0]:
# ---- verify_uploads.py ----
print("=== PDFs ===")
for f in os.listdir(PDF_DIR):
    print(f"  {f}")

print("\n=== JSONs ===")
for f in os.listdir(JSON_DIR):
    print(f"  {f}")

=== PDFs ===
  Canada.pdf
  FTP2023_Chapter01 (1).pdf
  FTP2023_Chapter01.pdf
  FTP2023_Chapter05.pdf
  FTP2023_Chapter07.pdf
  FTP2023_Chapter08.pdf
  FTP2023_Chapter09.pdf
  HBP Chapter 2 .pdf
  HBP Chapter 5.pdf
  HBP2023_Chapter01.pdf
  HBP2023_Chapter08.pdf
  HBP2023_Chapter09.pdf
  bahrain.pdf
  bangladesh.pdf
  bolivia.pdf
  cambodia.pdf
  greece.pdf
  indian_export_base_guidelines.pdf
  indonesia.pdf
  japan.pdf
  japan2.pdf

=== JSONs ===
  afghanistan.json
  argentina.json
  australia.json
  azerbaijan.json
  bahrain.json
  bangladesh.json
  china.json


In [0]:
# ---- load_pdfs.py ----
def load_pdfs():
    loader = DirectoryLoader(
        PDF_DIR,
        glob="**/*.pdf",
        loader_cls=PyMuPDFLoader
    )
    docs = loader.load()
    print(f"✅ Loaded {len(docs)} PDF pages from {len(set([d.metadata.get('source') for d in docs]))} files")
    return docs

pdf_docs = load_pdfs()

✅ Loaded 291 PDF pages from 21 files


In [0]:
# ---- load_jsons.py ----
def load_jsons():
    docs = []
    for file in os.listdir(JSON_DIR):
        if file.endswith(".json"):
            with open(f"{JSON_DIR}/{file}") as f:
                data = json.load(f)
                docs.append(Document(
                    page_content=json.dumps(data, indent=2),
                    metadata={
                        "source": file,
                        "type"  : "country_guideline"
                    }
                ))
    print(f"✅ Loaded {len(docs)} JSON files")
    return docs

json_docs = load_jsons()

✅ Loaded 7 JSON files


In [0]:
# ---- chunk_documents.py ----
def chunk_documents(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150
    )
    chunks = splitter.split_documents(docs)
    print(f"✅ Total chunks: {len(chunks)}")
    return chunks

all_docs = pdf_docs + json_docs
print(f"Total docs before chunking: {len(all_docs)}")
chunks = chunk_documents(all_docs)

Total docs before chunking: 298
✅ Total chunks: 811


In [0]:
# ---- create_delta_table.py ----
spark = SparkSession.builder.getOrCreate()

rows = []
for i, chunk in enumerate(chunks):
    rows.append({
        "id"     : str(i),
        "content": chunk.page_content,
        "source" : chunk.metadata.get("source", "unknown"),
        "type"   : chunk.metadata.get("type", "pdf")
    })

df_spark = spark.createDataFrame(pd.DataFrame(rows))

df_spark.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_NAME)

print(f"✅ Saved {len(rows)} chunks to {TABLE_NAME}")
display(spark.table(TABLE_NAME).limit(5))

✅ Saved 811 chunks to export_ai.rag.document_chunks


id,content,source,type
0,F&B Market in Canada SOME KEY INSIGHTS High Commission of India in Canada Ottawa,/Volumes/export_ai/rag/legal_docs/Canada.pdf,pdf
1,"Market Highlights  Population of 37 million limits the market size  Agriculture and Agro-processing are important components in Canada  Canada is a major food producer and exporter  Canadian F&B Retail market was worth $ 120 billion in 2020  F&B imports over $ 36 billion  Large immigrant population  Diversity of food choices  Ethnic Foods are a growing segment  Sophisticated Market  Quality Conscious and regulated  High Awareness  Canadian food processors rely on imported raw, semi-processed, and processed ingredients  A strong ‘buy local’ ethos  Major Multi-national brands are well-established in Canada  99% of the 6,500 F&B processing units in Canada are MSMEs  Demand for Healthy Foods  E-Commerce is growing  Import of food items from India is growing",/Volumes/export_ai/rag/legal_docs/Canada.pdf,pdf
2,"Post-Pandemic Picture  E-commerce is here to stay  93% of Canadians plan to either do more online shopping or the same amount post-pandemic.  Rising food costs will affect consumer behavior in terms of what – and where – they purchase food.  Interest in vegan products continues to climb.  At the same time, 78% of Canadians are yearning for comfort food after pandemic stress.  The health and wellness trend has shifted from niche to mainstream. Based on TFO Surveys and Reports",/Volumes/export_ai/rag/legal_docs/Canada.pdf,pdf
3,"Market Break-up Except Fresh Fruits and Vegetable Oils, United States has more than 60% share in all categories",/Volumes/export_ai/rag/legal_docs/Canada.pdf,pdf
4,Major Canadian Players (Revenues reflect 2018 figures),/Volumes/export_ai/rag/legal_docs/Canada.pdf,pdf


In [0]:
# ---- enable_cdf.py ----
spark.sql(f"""
    ALTER TABLE {TABLE_NAME}
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
print("✅ Change Data Feed enabled")

✅ Change Data Feed enabled


In [0]:
# ---- create_endpoint.py ----
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

try:
    vsc.create_endpoint(
        name          = ENDPOINT_NAME,
        endpoint_type = "STANDARD"
    )
    print("✅ Endpoint creation started")
except Exception as e:
    print(f"ℹ️ {e}")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True to VectorSearchClient().


ℹ️ Response content b'{"error_code":"QUOTA_EXCEEDED","message":"Maximum number of vector search endpoints per workspace exceeded quota of 1.","details":[{"@type":"type.googleapis.com/google.rpc.RequestInfo","request_id":"057b5a3a-ee9c-4c5b-9f26-f024fa54a08c","serving_data":""}]}', status_code 400


In [0]:
# ---- wait_for_endpoint.py ----
while True:
    endpoint = vsc.get_endpoint(name=ENDPOINT_NAME)
    status   = endpoint.get("endpoint_status", {}).get("state", "UNKNOWN")
    print(f"⏳ Endpoint status: {status}")

    if status == "ONLINE":
        print("✅ Endpoint is ONLINE — ready to proceed!")
        break
    elif status in ["OFFLINE", "FAILED"]:
        print("❌ Endpoint failed")
        break
    else:
        time.sleep(30)

⏳ Endpoint status: ONLINE
✅ Endpoint is ONLINE — ready to proceed!


In [0]:
# ---- create_vector_index.py ----
try:
    index = vsc.create_delta_sync_index(
        endpoint_name                 = ENDPOINT_NAME,
        index_name                    = INDEX_NAME,
        source_table_name             = TABLE_NAME,
        pipeline_type                 = "TRIGGERED",
        primary_key                   = "id",
        embedding_source_column       = "content",
        embedding_model_endpoint_name = "databricks-gte-large-en"
    )
    print("✅ Vector index creation started")
    print(index)
except Exception as e:
    print(f"ℹ️ {e}")

✅ Vector index creation started


In [0]:
# ---- wait_for_index.py ----
while True:
    index  = vsc.get_index(
        endpoint_name = ENDPOINT_NAME,
        index_name    = INDEX_NAME
    )
    
    # Correct way to get status
    status = index.describe().get("status", {})
    ready  = status.get("ready", False)
    state  = status.get("index_url", "provisioning")
    
    print(f"⏳ Index status: {status}")

    if ready:
        print("✅ Index is READY!")
        break
    else:
        print("Waiting 30 seconds...")
        time.sleep(30)

⏳ Index status: {'detailed_state': 'PROVISIONING_ENDPOINT', 'message': 'Delta sync index creation is pending endpoint provisioning.', 'ready': False, 'index_url': 'dbc-21bb95c1-0f91.cloud.databricks.com/api/2.0/vector-search/indexes/export_ai.rag.document_chunks_index'}
Waiting 30 seconds...
⏳ Index status: {'detailed_state': 'PROVISIONING_ENDPOINT', 'message': 'Delta sync index creation is pending endpoint provisioning.', 'ready': False, 'index_url': 'dbc-21bb95c1-0f91.cloud.databricks.com/api/2.0/vector-search/indexes/export_ai.rag.document_chunks_index'}
Waiting 30 seconds...
⏳ Index status: {'detailed_state': 'PROVISIONING_ENDPOINT', 'message': 'Delta sync index creation is pending endpoint provisioning.', 'ready': False, 'index_url': 'dbc-21bb95c1-0f91.cloud.databricks.com/api/2.0/vector-search/indexes/export_ai.rag.document_chunks_index'}
Waiting 30 seconds...
⏳ Index status: {'detailed_state': 'PROVISIONING_ENDPOINT', 'message': 'Delta sync index creation is pending endpoint pro

In [0]:
# ---- load_vectorstore.py ----
from langchain_databricks import DatabricksVectorSearch

ENDPOINT_NAME = "export_ai_vector_endpoint"
INDEX_NAME    = "export_ai.rag.document_chunks_index"

vectorstore = DatabricksVectorSearch(
    endpoint   = ENDPOINT_NAME,
    index_name = INDEX_NAME,
    columns    = ["content", "source", "type"]
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print("✅ Retriever ready")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True to VectorSearchClient().
✅ Retriever ready


In [0]:
# ---- verify_vectorstore.py ----
test_queries = [
    "What documents are needed to export to Bahrain?",
    "What are the legal requirements for food export?",
    "Which certificates are required for exporting spices?"
]

for query in test_queries:
    print(f"\n🔍 Query: {query}")
    results = retriever.invoke(query)
    for i, r in enumerate(results):
        print(f"  Result {i+1} | Source: {r.metadata.get('source')}")
        print(f"  Preview: {r.page_content[:200]}")
        print()


🔍 Query: What documents are needed to export to Bahrain?
  Result 1 | Source: bahrain.json
  Preview: {
  "country": "bahrain",
  "bahrain_import": {
    "overview": "Imports into Bahrain include goods transported via air, sea, land, courier, or postal services. The process is relatively streamlined w

  Result 2 | Source: bahrain.json
  Preview: "Sea freight",
        "Land transport",
        "Courier services",
        "Postal services"
      ]
    },
    "duties_and_charges": {
      "description": "Customs duties apply to imported goods."

  Result 3 | Source: /Volumes/export_ai/rag/legal_docs/bahrain.pdf
  Preview: 16 
Food Importers Guide to Kingdom 
 The name and address of the exporting  company 
 Brief description of the exported foodstuffs, their quantity and origin. 
 An attached list, authorized by the

  Result 4 | Source: azerbaijan.json
  Preview: {
  "country": "Azerbaijan",
  "documents_required": [
    "Signed import contract (with contract number)",
    "Customs